In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml

pd.set_option("display.max_columns", None)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from preprocessing import three_way_split, build_preprocessor, split_column_types, save_split

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
INTERIM_DIR = os.path.join(PROJECT_ROOT, config["paths"]["interim_dir"])
PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
TRAIN_FRAC = config["split"]["train_frac"]
CAL_FRAC = config["split"]["calibration_frac"]
TEST_FRAC = config["split"]["test_frac"]

In [2]:
from preprocessing import three_way_group_split

pk = pd.read_csv(os.path.join(INTERIM_DIR, "pakistan_interim.csv"))

sdoh_cols_pk = ["Age", "Gestational Age", "Number of sons ", "Number of daughters",
                "Total Number of Children", "Gravida", "Female Education", "Husband Education",
                "Working Status", "Physical Health ", "Previous Miscarriage",
                "Sufficient Money for Basic Needs", "Current Appereance Acceptance",
                "Family System", "Male Gender Preference", "Relationship with Mother in-law"]

model_cols_pk = sdoh_cols_pk + ["y_depression"]
pk_model = pk[model_cols_pk].copy()

train_pk, cal_pk, test_pk = three_way_group_split(
    pk_model, "y_depression", sdoh_cols_pk, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
print(train_pk.shape, cal_pk.shape, test_pk.shape)
print("Label balance:", train_pk["y_depression"].mean(), cal_pk["y_depression"].mean(), test_pk["y_depression"].mean())

num_cols_pk, cat_cols_pk = split_column_types(pk_model, sdoh_cols_pk)

(8402, 17) (2808, 17) (2798, 17)
Label balance: 0.653058795524875 0.6438746438746439 0.637240886347391


In [3]:
pre_pk = build_preprocessor(num_cols_pk, cat_cols_pk, scale_numeric=False)
X_check = pre_pk.fit_transform(train_pk[sdoh_cols_pk])
print("Transformed training shape:", X_check.shape)
print("Any NaN after preprocessing:", np.isnan(X_check).any())

Transformed training shape: (8402, 36)
Any NaN after preprocessing: False


In [4]:
save_split(train_pk, cal_pk, test_pk, PROCESSED_DIR, "pakistan")

Saved pakistan: train=(8402, 17), calibration=(2808, 17), test=(2798, 17)
  -> C:\maternal-mental-health-risk\data/processed\pakistan


In [5]:
ar = pd.read_csv(os.path.join(INTERIM_DIR, "arab5_interim.csv"))

sdoh_cols_ar = ["part1_country", "Locality_first", "WomenAge", "MariageAge", "NumberofPregnancy",
                "NumberofAbortions", "EducationLevel", "Work", "CovidDiagL1", "CovidDiagL2",
                "Health_Prob", "StressduringCovid", "FamilyProblems", "FinancialProblem",
                "SocialProblem", "PsychologicalProb", "WorkStress", "FamilyIncome1",
                "SleepingAfter", "BMI_Pregnancy", "part1_current_preg_first"]

# Excluded for heavy missingness (>40%), not leakage: BMI_Postpartum, Food_Group_Adher_During_CAT,
# Physical_activity_During_Cat, Smoker_During_preg_Cat
print("Missingness check on chosen predictors:")
print(ar[sdoh_cols_ar].isna().sum())

Missingness check on chosen predictors:
part1_country                 0
Locality_first                0
WomenAge                      0
MariageAge                    0
NumberofPregnancy             2
NumberofAbortions             4
EducationLevel                0
Work                          0
CovidDiagL1                   0
CovidDiagL2                   0
Health_Prob                   0
StressduringCovid             0
FamilyProblems                0
FinancialProblem              0
SocialProblem                 0
PsychologicalProb             0
WorkStress                    0
FamilyIncome1                 0
SleepingAfter                 0
BMI_Pregnancy               537
part1_current_preg_first      0
dtype: int64


In [6]:
from preprocessing import three_way_group_split

ar_dep = ar[ar["dep_items_complete"] == True].copy()
model_cols_dep = sdoh_cols_ar + ["y_depression"]
ar_dep_model = ar_dep[model_cols_dep].copy()

train_ar_dep, cal_ar_dep, test_ar_dep = three_way_group_split(
    ar_dep_model, "y_depression", sdoh_cols_ar, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
print(train_ar_dep.shape, cal_ar_dep.shape, test_ar_dep.shape)
print("Label balance:", train_ar_dep["y_depression"].mean(), cal_ar_dep["y_depression"].mean(), test_ar_dep["y_depression"].mean())

num_cols_ar, cat_cols_ar = split_column_types(ar_dep_model, sdoh_cols_ar)
pre_ar = build_preprocessor(num_cols_ar, cat_cols_ar, scale_numeric=False)
X_check = pre_ar.fit_transform(train_ar_dep[sdoh_cols_ar])
print("Transformed shape:", X_check.shape, "| any NaN:", np.isnan(X_check).any())

save_split(train_ar_dep, cal_ar_dep, test_ar_dep, PROCESSED_DIR, "arab5_depression")

(902, 22) (296, 22) (301, 22)
Label balance: 0.4445676274944568 0.5101351351351351 0.43853820598006643
Transformed shape: (902, 21) | any NaN: False
Saved arab5_depression: train=(902, 22), calibration=(296, 22), test=(301, 22)
  -> C:\maternal-mental-health-risk\data/processed\arab5_depression


In [7]:
ar_anx = ar[ar["anx_items_complete"] == True].copy()
model_cols_anx = sdoh_cols_ar + ["y_anxiety"]
ar_anx_model = ar_anx[model_cols_anx].copy()

train_ar_anx, cal_ar_anx, test_ar_anx = three_way_group_split(
    ar_anx_model, "y_anxiety", sdoh_cols_ar, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
print(train_ar_anx.shape, cal_ar_anx.shape, test_ar_anx.shape)

save_split(train_ar_anx, cal_ar_anx, test_ar_anx, PROCESSED_DIR, "arab5_anxiety")

(565, 22) (190, 22) (191, 22)
Saved arab5_anxiety: train=(565, 22), calibration=(190, 22), test=(191, 22)
  -> C:\maternal-mental-health-risk\data/processed\arab5_anxiety


In [8]:
model_cols_dep_full = sdoh_cols_ar + ["y_depression"]
ar_dep_full = ar[model_cols_dep_full].copy()
train_f, cal_f, test_f = three_way_group_split(
    ar_dep_full, "y_depression", sdoh_cols_ar, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
save_split(train_f, cal_f, test_f, PROCESSED_DIR, "arab5_depression_fullN_sensitivity")

Saved arab5_depression_fullN_sensitivity: train=(2132, 22), calibration=(719, 22), test=(718, 22)
  -> C:\maternal-mental-health-risk\data/processed\arab5_depression_fullN_sensitivity


In [11]:
ug = pd.read_csv(os.path.join(INTERIM_DIR, "uganda_interim.csv"))

sdoh_cols_ug = ["age_numeric", "homeDistrict", "occupation", "relationshipStatus",
                "spouseOccupation", "polygamy", "otherChildren", "numStillBorn",
                "numMissCarry", "education"]

model_cols_ug = sdoh_cols_ug + ["y_anxiety_binary", "Anxiety", "Total Score"]
ug_model = ug[model_cols_ug].copy()

train_ug, cal_ug, test_ug = three_way_group_split(
    ug_model, "y_anxiety_binary", sdoh_cols_ug, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
print(train_ug.shape, cal_ug.shape, test_ug.shape)
print("Label balance:", train_ug["y_anxiety_binary"].mean(), cal_ug["y_anxiety_binary"].mean(), test_ug["y_anxiety_binary"].mean())

num_cols_ug, cat_cols_ug = split_column_types(ug_model, sdoh_cols_ug)
pre_ug = build_preprocessor(num_cols_ug, cat_cols_ug, scale_numeric=False)
X_check = pre_ug.fit_transform(train_ug[sdoh_cols_ug])
print("Transformed shape:", X_check.shape, "| any NaN:", np.isnan(X_check).any())

save_split(train_ug, cal_ug, test_ug, PROCESSED_DIR, "uganda")

(8612, 13) (2872, 13) (2841, 13)
Label balance: 0.49431026474686485 0.487116991643454 0.4808166138683562
Transformed shape: (8612, 50) | any NaN: False
Saved uganda: train=(8612, 13), calibration=(2872, 13), test=(2841, 13)
  -> C:\maternal-mental-health-risk\data/processed\uganda


In [12]:
pn = pd.read_csv(os.path.join(INTERIM_DIR, "postnatal_interim.csv"))

predictor_items_enc = ["Irritable towards baby & partner_enc", "Trouble sleeping at night_enc",
                        "Problems concentrating or making decision_enc",
                        "Overeating or loss of appetite_enc", "Problems of bonding with baby_enc"]
sdoh_cols_pn = predictor_items_enc + ["Age"]

model_cols_pn = sdoh_cols_pn + ["y_ppd_risk"]
pn_model = pn[model_cols_pn].copy()

train_pn, cal_pn, test_pn = three_way_group_split(
    pn_model, "y_ppd_risk", sdoh_cols_pn, TRAIN_FRAC, CAL_FRAC, TEST_FRAC, SEED)
print(train_pn.shape, cal_pn.shape, test_pn.shape)
print("Label balance:", train_pn["y_ppd_risk"].mean(), cal_pn["y_ppd_risk"].mean(), test_pn["y_ppd_risk"].mean())

num_cols_pn, cat_cols_pn = split_column_types(pn_model, sdoh_cols_pn)
pre_pn = build_preprocessor(num_cols_pn, cat_cols_pn, scale_numeric=False)
X_check = pre_pn.fit_transform(train_pn[sdoh_cols_pn])
print("Transformed shape:", X_check.shape, "| any NaN:", np.isnan(X_check).any())

save_split(train_pn, cal_pn, test_pn, PROCESSED_DIR, "postnatal")

(959, 7) (311, 7) (233, 7)
Label balance: 0.4807090719499479 0.4855305466237942 0.41630901287553645
Transformed shape: (959, 10) | any NaN: False
Saved postnatal: train=(959, 7), calibration=(311, 7), test=(233, 7)
  -> C:\maternal-mental-health-risk\data/processed\postnatal


In [5]:
for name in ["pakistan", "arab5_depression", "arab5_anxiety",
             "arab5_depression_fullN_sensitivity", "uganda", "postnatal"]:
    folder = os.path.join(PROCESSED_DIR, name)
    files = os.listdir(folder)
    print(f"{name}: {files}")

pakistan: ['base_predictions_calibration.csv', 'base_predictions_test.csv', 'calibration.csv', 'oof_predictions_train.csv', 'test.csv', 'train.csv']
arab5_depression: ['base_predictions_calibration.csv', 'base_predictions_test.csv', 'calibration.csv', 'oof_predictions_train.csv', 'test.csv', 'train.csv']
arab5_anxiety: ['base_predictions_calibration.csv', 'base_predictions_test.csv', 'calibration.csv', 'oof_predictions_train.csv', 'test.csv', 'train.csv']
arab5_depression_fullN_sensitivity: ['base_predictions_calibration.csv', 'base_predictions_test.csv', 'calibration.csv', 'oof_predictions_train.csv', 'test.csv', 'train.csv']
uganda: ['base_predictions_calibration.csv', 'base_predictions_test.csv', 'calibration.csv', 'oof_predictions_train.csv', 'test.csv', 'train.csv']
postnatal: ['calibration.csv', 'test.csv', 'train.csv']
